# 05 — Per-machine analysis

How time on each machine was split between productive processing,
maintenance (broken / waiting for repair), and how often each broke
down.  Identifies the bottleneck and reliability-sensitive machines.

In [ ]:
import sys, pathlib
# Make ``utils.py`` importable whether the notebook is opened from
# project root or from analysis/.
_here = pathlib.Path.cwd()
for cand in (_here, _here / "analysis", _here.parent):
    if (cand / "utils.py").exists():
        sys.path.insert(0, str(cand))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import (
    list_runs, latest_run, load_run, load_runs,
    group_columns, group_columns_by_prefix, strip_prefix, rolling_mean,
)

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

In [ ]:
# Pick a run.  ``latest_run()`` returns the most recently modified
# experiment in ``reports/``.  Override ``EXP_ID = "..."`` to analyse
# a specific one.
EXP_ID = latest_run()
print("Analysing:", EXP_ID)

ep, st, cfg = load_run(EXP_ID)
print(f"  episode_metrics.csv: shape={ep.shape}")
print(f"  step_metrics.csv:    shape={st.shape if st is not None else 'absent'}")

## Total maintenance / production time per machine across the run

In [ ]:
train = ep[ep["phase"] == "train"]

maint = train[group_columns(train, "machine_maintenance")].sum()
prod  = train[group_columns(train, "machine_production")].sum()
brk   = train[group_columns(train, "machine_breakdowns")].sum()
for s, p in ((maint, "machine_maintenance"), (prod, "machine_production"), (brk, "machine_breakdowns")):
    s.index = strip_prefix(list(s.index), p)

if not maint.empty:
    df = pd.concat([prod, maint], axis=1, keys=["production", "maintenance"]).fillna(0)
    df = df.sort_values("maintenance", ascending=True)
    fig, ax = plt.subplots(figsize=(11, max(4, 0.32 * len(df))))
    df.plot.barh(stacked=True, ax=ax, color=["C2", "C3"], edgecolor="black", linewidth=0.3)
    ax.set(xlabel="cumulative sim time (across episodes)", title="Time-on-machine breakdown")
    plt.tight_layout(); plt.show()

## Breakdowns per machine

In [ ]:
if not brk.empty:
    s = brk.sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(10, max(3, 0.32 * len(s))))
    s.plot.barh(ax=ax, color="C3", edgecolor="black", linewidth=0.3)
    ax.invert_yaxis()
    ax.set(xlabel="total breakdowns (across episodes)", title="Reliability — breakdowns per machine")
    plt.tight_layout(); plt.show()

## Utilisation = production / (production + maintenance)

In [ ]:
if not maint.empty:
    util = prod / (prod + maint).replace(0, np.nan)
    util = util.sort_values()
    fig, ax = plt.subplots(figsize=(10, max(3, 0.32 * len(util))))
    util.plot.barh(ax=ax, color="C0", edgecolor="black", linewidth=0.3)
    ax.set(xlabel="production / (production + maintenance)", title="Per-machine utilisation",
           xlim=(0, 1))
    plt.tight_layout(); plt.show()